In [4]:
# import libraries

import logging
from boto3.session import Session


# Import AgentCore Memory client and Hook events from strands
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType
from strands.hooks import (
    AfterInvocationEvent,
    HookProvider,
    HookRegistry,
    MessageAddedEvent,
)

from lab_helpers.utils import get_ssm_parameter, put_ssm_parameter

boto_session = Session()
REGION = boto_session.region_name

logger = logging.getLogger(__name__)

In [5]:
# Creating memory resources involves provisioning the underlying infrastructure (vector databases, processing pipelines, etc.). 
# This typically takes 2-3 minutes as AWS sets up the managed services behind the scenes.
memory_client = MemoryClient(region_name=REGION)
memory_name = "CustomerSupportMemory"


def create_or_get_memory_resource():
    try:
        memory_id = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
        memory_client.gmcp_client.get_memory(memoryId=memory_id)
        return memory_id
    except Exception:
        try:
            strategies = [
                {
                    StrategyType.USER_PREFERENCE.value: {
                        "name": "CustomerPreferences",
                        "description": "Captures customer preferences and behavior",
                        "namespaces": ["support/customer/{actorId}/preferences"],
                    }
                },
                {
                    StrategyType.SEMANTIC.value: {
                        "name": "CustomerSupportSemantic",
                        "description": "Stores facts from conversations",
                        "namespaces": ["support/customer/{actorId}/semantic"],
                    }
                },
            ]
            print("Creating AgentCore Memory resources. This will take 2-3 minutes...")
            print("While we wait, let's understand what's happening behind the scenes:")
            print("• Setting up managed vector databases for semantic search")
            print("• Configuring memory extraction pipelines")
            print("• Provisioning secure, multi-tenant storage")
            print("• Establishing namespace isolation for customer data")
            # *** AGENTCORE MEMORY USAGE *** - Create memory resource with semantic strategy
            response = memory_client.create_memory_and_wait(
                name=memory_name,
                description="Customer support agent memory",
                strategies=strategies,
                event_expiry_days=90,  # Memories expire after 90 days
            )
            memory_id = response["id"]
            try:
                put_ssm_parameter("/app/customersupport/agentcore/memory_id", memory_id)
            except Exception as e:
                raise e
            return memory_id
        except Exception as e:
            print(f"Failed to create memory resource: {e}")
            return None

In [6]:
memory_id = create_or_get_memory_resource()
if memory_id:
    print("✅ AgentCore Memory created successfully!")
    print(f"Memory ID: {memory_id}")
else:
    print("Memory resource not created. Try Again !")

Creating AgentCore Memory resources. This will take 2-3 minutes...
While we wait, let's understand what's happening behind the scenes:
• Setting up managed vector databases for semantic search
• Configuring memory extraction pipelines
• Provisioning secure, multi-tenant storage
• Establishing namespace isolation for customer data
✅ AgentCore Memory created successfully!
Memory ID: CustomerSupportMemory-X72O7a3jPa


In [7]:
# List existing memory resources
for memory in memory_client.list_memories():
    print(f"Memory Arn: {memory.get('arn')}")
    print(f"Memory ID: {memory.get('id')}")
    print("--------------------------------------------------------------------")

# Seed with previous customer interactions
CUSTOMER_ID = "customer_001"

previous_interactions = [
    ("I'm having issues with my MacBook Pro overheating during video editing.", "USER"),
    (
        "I can help with that thermal issue. For video editing workloads, let's check your Activity Monitor and adjust performance settings. Your MacBook Pro order #MB-78432 is still under warranty.",
        "ASSISTANT",
    ),
    (
        "What's the return policy on gaming headphones? I need low latency for competitive FPS games",
        "USER",
    ),
    (
        "For gaming headphones, you have 30 days to return. Since you're into competitive FPS, I'd recommend checking the audio latency specs - most gaming models have <40ms latency.",
        "ASSISTANT",
    ),
    (
        "I need a laptop under $1200 for programming. Prefer 16GB RAM minimum and good Linux compatibility. I like ThinkPad models.",
        "USER",
    ),
    (
        "Perfect! For development work, I'd suggest looking at our ThinkPad E series or Dell XPS models. Both have excellent Linux support and 16GB RAM options within your budget.",
        "ASSISTANT",
    ),
]

# Save previous interactions
if memory_id:
    try:
        memory_client.create_event(
            memory_id=memory_id,
            actor_id=CUSTOMER_ID,
            session_id="previous_session",
            messages=previous_interactions,
        )
        print("✅ Seeded customer history successfully")
        print("📝 Interactions saved to Short-Term Memory")
        print("⏳ Long-Term Memory processing will begin automatically...")
    except Exception as e:
        print(f"⚠️ Error seeding history: {e}")

Memory Arn: arn:aws:bedrock-agentcore:us-west-2:956723944867:memory/CustomerSupportMemory-X72O7a3jPa
Memory ID: CustomerSupportMemory-X72O7a3jPa
--------------------------------------------------------------------
✅ Seeded customer history successfully
📝 Interactions saved to Short-Term Memory
⏳ Long-Term Memory processing will begin automatically...


In [8]:
import time

# Wait for Long-Term Memory processing to complete
print("🔍 Checking for processed Long-Term Memories...")
retries = 0
max_retries = 6  # 1 minute wait

while retries < max_retries:
    memories = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace=f"support/customer/{CUSTOMER_ID}/preferences",
        query="can you summarize the support issue",
    )

    if memories:
        print(
            f"✅ Found {len(memories)} preference memories after {retries * 10} seconds!"
        )
        break

    retries += 1
    if retries < max_retries:
        print(
            f"⏳ Still processing... waiting 10 more seconds (attempt {retries}/{max_retries})"
        )
        time.sleep(10)
    else:
        print(
            "⚠️ Memory processing is taking longer than expected. This can happen with overloading.."
        )
        break

print(
    "🎯 AgentCore Memory automatically extracted these customer preferences from our seeded conversations:"
)
print("=" * 80)

for i, memory in enumerate(memories, 1):
    if isinstance(memory, dict):
        content = memory.get("content", {})
        if isinstance(content, dict):
            text = content.get("text", "")
            print(f"  {i}. {text}")

🔍 Checking for processed Long-Term Memories...
✅ Found 2 preference memories after 0 seconds!
🎯 AgentCore Memory automatically extracted these customer preferences from our seeded conversations:
  1. {"context":"Selecting gaming headphones","preference":"Needs low latency audio for competitive FPS games","categories":["gaming headphones","audio latency","FPS games"]}
  2. {"context":"Purchasing a laptop for programming","preference":"Needs a laptop under $1200 with at least 16GB RAM, good Linux compatibility, and prefers ThinkPad models","categories":["laptops","programming","Linux","RAM","budget","ThinkPad"]}


In [9]:
import time

# Retrieve semantic memories (factual information)
print("🔍 Checking for processed Semantic Memories...")
retries = 0
max_retries = 6  # 1 minute wait

while retries < max_retries:
    semantic_memories = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace=f"support/customer/{CUSTOMER_ID}/semantic",
        query="information on the technical support issue",
    )

    if semantic_memories:
        print(f"✅ Found {len(semantic_memories)} semantic memories after {retries * 10} seconds!")
        break

    retries += 1
    if retries < max_retries:
        print(f"⏳ Still processing... waiting 10 more seconds (attempt {retries}/{max_retries})")
        time.sleep(10)
    else:
        print("⚠️ Semantic memory processing is taking longer than expected.")
        break

print("🧠 AgentCore Memory identified these factual details from conversations:")
print("=" * 80)

for i, memory in enumerate(semantic_memories, 1):
    if isinstance(memory, dict):
        content = memory.get("content", {})
        if isinstance(content, dict):
            text = content.get("text", "")
            print(f"  {i}. {text}")

🔍 Checking for processed Semantic Memories...
✅ Found 3 semantic memories after 0 seconds!
🧠 AgentCore Memory identified these factual details from conversations:
  1. The user has a MacBook Pro (order #MB-78432) that is still under warranty.
  2. The user plays competitive FPS games and requires low latency audio.
  3. The user experiences overheating issues with their MacBook Pro during video editing.


In [10]:
# architectural class of memory agent
class CustomerSupportMemoryHooks(HookProvider):
    """Memory hooks for customer support agent"""

    def __init__(
        self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str
    ):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def retrieve_customer_context(self, event: MessageAddedEvent):
        """Retrieve customer context before processing support query"""
        messages = event.agent.messages
        if (
            messages[-1]["role"] == "user"
            and "toolResult" not in messages[-1]["content"][0]
        ):
            user_query = messages[-1]["content"][0]["text"]

            try:
                all_context = []

                for context_type, namespace in self.namespaces.items():
                    # *** AGENTCORE MEMORY USAGE *** - Retrieve customer context from each namespace
                    memories = self.client.retrieve_memories(
                        memory_id=self.memory_id,
                        namespace=namespace.format(actorId=self.actor_id),
                        query=user_query,
                        top_k=3,
                    )
                    # Post-processing: Format memories into context strings
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(
                                        f"[{context_type.upper()}] {text}"
                                    )

                # Inject customer context into the query
                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0]["text"] = (
                        f"Customer Context:\n{context_text}\n\n{original_text}"
                    )
                    logger.info(f"Retrieved {len(all_context)} customer context items")

            except Exception as e:
                logger.error(f"Failed to retrieve customer context: {e}")

    def save_support_interaction(self, event: AfterInvocationEvent):
        """Save customer support interaction after agent response"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Get last customer query and agent response
                customer_query = None
                agent_response = None

                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif (
                        msg["role"] == "user"
                        and not customer_query
                        and "toolResult" not in msg["content"][0]
                    ):
                        customer_query = msg["content"][0]["text"]
                        break

                if customer_query and agent_response:
                    # *** AGENTCORE MEMORY USAGE *** - Save the support interaction
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[
                            (customer_query, "USER"),
                            (agent_response, "ASSISTANT"),
                        ],
                    )
                    logger.info("Saved support interaction to memory")

        except Exception as e:
            logger.error(f"Failed to save support interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register customer support memory hooks"""
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)
        logger.info("Customer support memory hooks registered")

In [12]:
 #  implement the Customer Support Agent just as we did in Lab 1, but this time we instantiate the class CustomerSupportMemoryHooks and we pass 
# the memory hook to the agent contructor.

import uuid

from strands import Agent
from strands.models import BedrockModel

from lab_helpers.lab1_strands_agent import (
    SYSTEM_PROMPT,
    get_return_policy,
    web_search,
    get_product_info,
    get_technical_support,
    MODEL_ID,
)

SESSION_ID = str(uuid.uuid4())
memory_hooks = CustomerSupportMemoryHooks(
    memory_id, memory_client, CUSTOMER_ID, SESSION_ID
)


# Initialize the Bedrock model (Anthropic Claude 3.7 Sonnet)
model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

# Create the customer support agent with all 5 tools
agent = Agent(
    model=model,
    hooks=[memory_hooks],  # Pass Memory Hooks
    tools=[
        get_product_info,  # Tool 1: Simple product information lookup
        get_return_policy,  # Tool 2: Simple return policy lookup
        web_search,
        get_technical_support,
    ],
    system_prompt=SYSTEM_PROMPT,
)

In [13]:
# Let's test our memory-enhanced agent! Watch how it uses the customer's historical preferences to provide personalized recommendations.

# The agent will automatically:

# Retrieve relevant customer context from memory
# Use that context to personalize the response
# Save this new interaction for future use

print("🎧 Testing headphone recommendation with customer memory...\n\n")
response1 = agent("Which headphones would you recommend?")

🎧 Testing headphone recommendation with customer memory...



Tool #1: get_product_info
I understand you're looking for gaming headphones with low latency audio specifically for competitive FPS games. However, I don't have access to the technical specifications for gaming headphones at the moment.

Let me get you in touch with our technical support team who can provide detailed information about gaming headphones with low latency audio that's suitable for competitive FPS gaming. They can help you find models that meet your specific requirements for minimal audio delay.

Would you like me to connect you with our technical support team right away? They can provide specific recommendations for gaming headphones that will give you the responsive audio experience you need for competitive FPS games.

In [14]:
print("\n💻 Testing laptop preference recall...\n\n")
response2 = agent("What is my preferred laptop brand and requirements?")


💻 Testing laptop preference recall...


Based on your context, your preferred laptop brand is **ThinkPad**, and here are your specific requirements for a programming laptop:

**Your Laptop Requirements:**
- **Brand Preference:** ThinkPad models
- **Budget:** Under $1,200
- **RAM:** Minimum of 16GB
- **OS Compatibility:** Good Linux compatibility
- **Use Case:** Programming

You're specifically looking for a ThinkPad laptop that fits within your $1,200 budget, has at least 16GB of RAM, and works well with Linux for your programming needs.

Would you like me to help you find ThinkPad models that meet these specifications? I can search for current ThinkPad options that match your requirements.

In [ ]:
print("\n💻 Testing laptop preference recall...\n\n")
response2 = agent("Can I use the headphones with apple prod?")